## Regressão múltipla — `valor_aluguel ~ distancia_praia_km + area_m2 + quartos`

Mesma base do notebook de regressão linear simples (`bases/aluguel_praia_polinomial.csv`),
agora usando os três atributos disponíveis. A formulação continua sendo uma reta
(mínimos quadrados) — só que agora com mais de um coeficiente para interpretar.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEMENTE = 42
pd.set_option("display.width", 100)

df = pd.read_csv("bases/aluguel_praia_1.csv")
print("shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'bases/aluguel_praia_1.csv'

### 1. Correlação entre os atributos e o alvo

In [ ]:
df.corr(numeric_only=True)["valor_aluguel"].sort_values(ascending=False)

**Leitura:** `distancia_praia_km` tem a correlação mais forte (negativa) com o
aluguel, mas é uma correlação LINEAR — ela não captura o quanto a relação real é
curva. `area_m2` e `quartos` têm correlações positivas mais modestas.

### 2. Separando treino e teste

Mesma divisão 70/30 e mesma semente do notebook de regressão simples, para que
os resultados sejam comparáveis linha a linha.

In [ ]:
colunas = ["distancia_praia_km", "area_m2", "quartos"]
X = df[colunas].values
y = df["valor_aluguel"].values

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=SEMENTE
)
print(f"treino: {X_treino.shape[0]} imoveis   teste: {X_teste.shape[0]} imoveis")

### 3. Treinando a regressão múltipla

`y_previsto = b0 + b1*distancia_praia_km + b2*area_m2 + b3*quartos`

In [ ]:
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

print(f"b0 (intercepto) = {modelo.intercept_:.2f}")
for nome, coef in zip(colunas, modelo.coef_):
    print(f"b (coeficiente de {nome}) = {coef:.2f}")

**Leitura:** o coeficiente de `distancia_praia_km` aqui é a inclinação de uma
ÚNICA reta tentando resumir uma relação que, na verdade, cai rápido e depois
achata — é uma média grosseira do efeito real, mantendo `area_m2` e `quartos`
constantes.

### 4. Avaliando no teste

In [ ]:
y_pred = modelo.predict(X_teste)

mae = mean_absolute_error(y_teste, y_pred)
rmse = mean_squared_error(y_teste, y_pred) ** 0.5
r2 = r2_score(y_teste, y_pred)

print(f"MAE  = R$ {mae:.2f}")
print(f"RMSE = R$ {rmse:.2f}")
print(f"R^2  = {r2:.4f}")

### 5. Comparando com a regressão linear simples (só `distancia_praia_km`)

In [ ]:
X_simples = df[["distancia_praia_km"]].values
Xs_treino, Xs_teste, ys_treino, ys_teste = train_test_split(
    X_simples, y, test_size=0.30, random_state=SEMENTE
)
modelo_simples = LinearRegression().fit(Xs_treino, ys_treino)
pred_simples = modelo_simples.predict(Xs_teste)

comparacao = pd.DataFrame({
    "modelo": ["Linear simples (distancia)", "Linear multipla (distancia+area+quartos)"],
    "MAE": [mean_absolute_error(ys_teste, pred_simples), mae],
    "RMSE": [mean_squared_error(ys_teste, pred_simples) ** 0.5, rmse],
    "R2": [r2_score(ys_teste, pred_simples), r2],
})
comparacao.round(2)

**Leitura:** acrescentar `area_m2` e `quartos` melhora bastante o ajuste em
relação a usar só `distancia_praia_km` — são atributos realmente informativos.
Mas vale a pena comparar este R² com o do notebook de regressão POLINOMIAL: lá,
usando **só** `distancia_praia_km` (menos informação, uma única variável), o
modelo pode superar esta múltipla — porque o que está sendo corrigido ali não é
a falta de atributos, é a forma errada (reta) usada para o efeito da distância.
Vale a pena rodar os dois notebooks e comparar os números de perto.

### 6. Previsto x real

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_teste, y_pred, s=14, alpha=0.4, color="#2c5aa0")
lims = [min(y_teste.min(), y_pred.min()), max(y_teste.max(), y_pred.max())]
ax.plot(lims, lims, color="#c0392b", linewidth=1.5, linestyle="--", label="previsto = real")
ax.set_xlabel("valor real (R$)")
ax.set_ylabel("valor previsto (R$)")
ax.set_title("Regressao multipla: previsto x real (teste)")
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

### 7. Pesquisar um valor

Digite distância até a praia, área e número de quartos para ver o aluguel
previsto por este modelo. Como no notebook de regressão simples, sem entrada
interativa disponível a célula usa valores de exemplo.

In [ ]:
def prever_valor_aluguel(distancia_km, area_m2, quartos):
    X_novo = np.array([[distancia_km, area_m2, quartos]])
    return modelo.predict(X_novo)[0]


valores_exemplo = dict(distancia_km=3.0, area_m2=70.0, quartos=2)
try:
    e1 = float(input("Digite a distancia da praia em km: ").replace(",", "."))
    e2 = float(input("Digite a area do imovel em m2: ").replace(",", "."))
    e3 = float(input("Digite o numero de quartos: ").replace(",", "."))
    entradas = dict(distancia_km=e1, area_m2=e2, quartos=e3)
except Exception:
    entradas = valores_exemplo
    print(f"(entrada interativa indisponivel - usando valores de exemplo: {valores_exemplo})")

valor_previsto = prever_valor_aluguel(**entradas)
print(f"\nEntradas: {entradas}")
print(f"Valor de aluguel previsto (regressao multipla): R$ {valor_previsto:.2f}")

### O que guardar deste notebook

- Comparar modelos exige usar a MESMA divisão treino/teste (mesma semente) —
  caso contrário parte da diferença de MAE/RMSE/R² pode vir só do sorteio dos
  dados.
- Acrescentar atributos realmente informativos melhora as métricas de uma
  regressão linear — mas isso é diferente de corrigir a FORMA da relação de um
  atributo específico (o que os notebooks de regressão polinomial e SVR fazem
  com `distancia_praia_km`).
- Os dois problemas (atributos faltando x forma errada) pedem soluções
  diferentes — vale sempre perguntar qual dos dois está limitando o modelo
  antes de decidir o que tentar em seguida.